<center><img src="./imge/dice.jpg" alt="启东汤森" style="width: 300px;"/></center>

# 6.OpenCV 定位物体的实时位置

@  文件名：6.OpenCV定位物体的实时位置.ipynb <br>
@  版本：V2.0 <br>
@  说明：OpenCV定位物体的实时位置<br>

In [1]:
### 调整机械臂到达Home 点，便于获取图像
import math
from arm5dof_uservo import Arm5DoFUServo
from config import *

In [2]:
### 机械臂创建
arm = Arm5DoFUServo()

In [3]:
# 回到Home 点
arm.home()

In [4]:
arm.uservo.set_servo_angle(0,88,interval=1000)
arm.uservo.set_servo_angle(3,-80,interval=1000)
arm.uservo.wait()

In [5]:
# 载入必要的库文件
from __future__ import print_function
from imutils.video import VideoStream
import imutils
import time
import cv2
import os
import RPi.GPIO as GPIO

In [6]:
# 蜂鸣器初始化
beep = 17
GPIO.setwarnings(False)
GPIO.setmode(GPIO.BCM)
GPIO.setup(beep, GPIO.OUT)

In [7]:
# 打印物体的实时位置
def mapObjectPosition (x, y):
    print ("[INFO] Object Center coordenates at X0 = {0} and Y0 =  {1}".format(x, y))

In [8]:
dispW=500
dispH=350
flip=4
#调用CSI摄像头的属性参数
# 初始化视频流并允许摄像机传感器预热
#print("[INFO] waiting for camera to warmup...")
vs = cv2.VideoCapture(0)
#time.sleep(2.0) 

In [9]:
# 定义对象的上下边界
# 在HSV颜色空间中进行跟踪
colorLower = (9,135,231)
colorUpper = (31,255,255)

In [10]:
# 创建显示控件

import enum
import cv2
from jetcam.utils import bgr8_to_jpeg

def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])
import traitlets
import threading
import ipywidgets.widgets as widgets
from IPython.display import display
Frame = widgets.Image(format='jpeg', width=500, height=350)
display(Frame)

Image(value=b'', format='jpeg', height='350', width='500')

In [11]:
# 线程函数操作库
import threading # 线程
import ctypes
import inspect

# 线程结束代码
def _async_raise(tid, exctype):
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        raise SystemError("PyThreadState_SetAsyncExc failed")
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)

In [12]:
# 关闭蜂鸣器
GPIO.output(beep, GPIO.LOW)
beepOn = False

In [13]:
def Video_display():
    global beepOn
    # 循环的帧从视频流
    while True:
        # 从视频流中抓取下一帧，调整大小
        # 帧，并将其转换为HSV颜色空间
        ret, frame = vs.read()
        frame = imutils.resize(frame, width=500)
        frame = imutils.rotate(frame, angle=0)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # 为对象颜色构造一个遮罩，然后执行
        # 一系列的膨胀和侵蚀，以消除任何小的
        # blobs left in the mask
        mask = cv2.inRange(hsv, colorLower, colorUpper)
        mask = cv2.erode(mask, None, iterations=2)
        mask = cv2.dilate(mask, None, iterations=2)

        # 找到遮罩中的轮廓并初始化
        # (x, y) center of the object
        cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[-2]
        #cnts = cnts[0] if imutils.is_cv2() else cnts[1]
        center = None

        # 只有在找到至少一条轮廓线时才进行
        if len(cnts) > 0:
            # 在蒙版中找到最大的轮廓，然后使用
            # 它可以计算出最小的围圆
            # 重心
            c = max(cnts, key=cv2.contourArea)
            ((x, y), radius) = cv2.minEnclosingCircle(c)
            M = cv2.moments(c)
            center = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))

            # 只有当半径满足最小尺寸时才进行
            if radius > 10:
                # 在框架上画圆和质心，
                # 然后更新跟踪点的列表
                cv2.circle(frame, (int(x), int(y)), int(radius),(0, 255, 255), 2)
                cv2.circle(frame, center, 5, (0, 0, 255), -1)

                # 定位舵机在圆心
                mapObjectPosition(int(x), int(y))

                # 如果led还没有打开，打开led
                if not beepOn:
                    GPIO.output(beep, GPIO.HIGH)
                    beepOn = True
        # 如果没有检测到球，关闭LED灯
        elif beepOn:
            GPIO.output(beep, GPIO.LOW)
            beepOn = False

        # 向我们的屏幕显示框架
        Frame.value = bgr8_to_jpeg(frame)    
    # 做点清理工作
    print("\n [INFO] Exiting Program and cleanup stuff \n")
    GPIO.cleanup()
    vs.stop()

In [14]:
# 开始线程
t = threading.Thread(target=Video_display)
t.setDaemon(True)
t.start()

[INFO] Object Center coordenates at X0 = 15 and Y0 =  157
[INFO] Object Center coordenates at X0 = 48 and Y0 =  141
[INFO] Object Center coordenates at X0 = 79 and Y0 =  131
[INFO] Object Center coordenates at X0 = 100 and Y0 =  119
[INFO] Object Center coordenates at X0 = 79 and Y0 =  107
[INFO] Object Center coordenates at X0 = 115 and Y0 =  128
[INFO] Object Center coordenates at X0 = 175 and Y0 =  175
[INFO] Object Center coordenates at X0 = 215 and Y0 =  204
[INFO] Object Center coordenates at X0 = 231 and Y0 =  231
[INFO] Object Center coordenates at X0 = 243 and Y0 =  260
[INFO] Object Center coordenates at X0 = 242 and Y0 =  272
[INFO] Object Center coordenates at X0 = 255 and Y0 =  283
[INFO] Object Center coordenates at X0 = 242 and Y0 =  292
[INFO] Object Center coordenates at X0 = 248 and Y0 =  276
[INFO] Object Center coordenates at X0 = 250 and Y0 =  259
[INFO] Object Center coordenates at X0 = 253 and Y0 =  350
[INFO] Object Center coordenates at X0 = 253 and Y0 =  347
[

In [15]:
# 结束线程
stop_thread(t)